# Parabolic SDF


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
import math
from typing import NamedTuple
import numpy as np
from numpy.random import random as nprand
from numpy.typing import NDArray
import ipywidgets as wg
import k3d

from utils import npvec, arr, arrgs, garr, unflat, unflat_np, f32, disquance, normalize

In [ ]:
def pladd(plot, *args):
    for a in args:
        plot.__iadd__(a)

## The Fucking Parabola

$$
y = a_2 x^2 + a_1 x + a_0
$$

$$S(t) = [t, a_2 t^2 + a_1 t + a_0]$$

$$\frac{dS}{dt}(t) = [1, 2 a_2 t + a_1]$$

Symmetric around $Y$: $a_1 = 0$

$$S(t) = [t, a_2 t^2 + a_0]$$

$$\frac{dS}{dt}(t) = [1, 2 a_2 t]$$


In [ ]:
class Parab(NamedTuple):
    a0: float
    a2: float

    def point(self, t: float) -> npvec:
        return arrgs(t, self.a0 + self.a2 * t * t)

    def curve(self, tspace: np.ndarray[tuple[int]]) -> NDArray:
        return garr(self.point(t) for t in tspace)

    def flow(self, t: float) -> npvec:
        return arrgs(1, 2 * self.a2 * t)

## The SDF

$$
(p - S(t))·\frac{dS}{dt}(t) = 0
\tag{perpendicularity}
$$

$$
[1, t, t^2, t^3]
\begin{pmatrix}
- 0.5 p_x \\
0.5 + a_2(a_0 - p_y) \\
0 \\
a_2^2 \\
\end{pmatrix} = 0
$$


In [ ]:
def solve_np(parab: Parab, sample: npvec) -> tuple[float, ...]:
    x = sample[0]
    y = sample[1]

    coefs = arrgs(
        parab.a2**2,  # a
        0,  # b
        0.5 + (parab.a0 - y) * parab.a2,  # c
        -0.5 * x,  # d
    )

    roots = np.roots(coefs)
    return tuple(r.real for r in roots if r.imag == 0.0)

In [ ]:
from math import sqrt, cbrt, acos, cos, pow, pi as PI


def solve(parab: Parab, sample: npvec) -> float:
    x = sample[0]
    y = sample[1]

    a22 = pow(parab.a2, 2)
    p3 = (0.5 + parab.a2 * (parab.a0 - y)) / (3 * a22)  # p/3
    q2 = abs(x) / (4 * a22)  # -q/2
    D = pow(q2, 2) + pow(p3, 3)

    if D > 0:
        C = cbrt(q2 + sqrt(D))
        t = C - p3 / C
    else:
        r = sqrt(-p3)
        th = acos(q2 / pow(r, 3)) / 3.0
        t = 2 * r * cos(th)

    return t if x >= 0 else -t

In [ ]:
def side(parab: Parab, sample: npvec, t: float, point: npvec) -> int:
    ort = sample - point
    tng = parab.flow(t)
    crz = ort[1] * tng[0] - ort[0] * tng[1]
    return int(np.sign(crz))

In [ ]:
class Projection(NamedTuple):
    smp: npvec
    t: float
    pnt: npvec
    disq: float = 0
    side: int = 0

    @property
    def sdist(self):
        return self.side * np.sqrt(self.disq)

In [ ]:
def project_np(parab: Parab, sample: npvec) -> Projection:
    tt = solve_np(parab, sample)
    pp = tuple(parab.point(t) for t in tt)
    dd = tuple(disquance(p, sample) for p in pp)
    projs = [Projection(sample, t, p, d, side(parab, sample, t, p)) for t, p, d in zip(tt, pp, dd)]
    projs.sort(key=lambda prj: prj.disq)
    return projs[0]

In [ ]:
def project(parab: Parab, sample: npvec):
    t = solve(parab, sample)
    p = parab.point(t)
    d = disquance(p, sample)
    return Projection(sample, t, p, d, side(parab, sample, t, p))

---


In [ ]:
def randomize():
    a0 = (nprand() - 0.5) * 0.5
    a2 = (nprand() - 0.5) * 4
    return Parab(a0, a2)

In [ ]:
parab = Parab(0, 1)
tspace = np.linspace(-1.0, +1.0, 16, dtype=np.float32)

# Plotting


In [ ]:
plot = k3d.Plot(
    height=720,
    background_color=0x404040,
    grid_color=0x383838,
    label_color=0x000000,
    menu_visibility=False,
    grid=(-1.0, -1.0, 0.0, 1.0, 1.0, 1.0),
    grid_auto_fit=False,
    mode="callback",
)
plot.layout = wg.Layout(width="720px", height="720px")
plot.camera_auto_fit = False
plot.camera = [0, 0, 5, 0, 1, 0, 0, 0, 0]

In [ ]:
randomize_btn = wg.Button(description="randomize")
sample_btn = wg.Button(description="sample")
toggle2 = wg.Checkbox(description="tangents", value=False)

In [ ]:
wg.HBox([plot, wg.VBox([toggle2, randomize_btn, sample_btn])], layout=dict(width="100%", grid_gap="8px"))

In [ ]:
k3curve = k3d.line(vertices=[], shader="mesh", color=0xF0F0F0, line_width=0.25, color_map=k3d.colormaps.matplotlib_color_maps.Rainbow, color_range=[-1.0, 1.0])
k3flow = k3d.vectors(origins=[(0, 0, 0)], vectors=[(0, 0, 0)], use_head=False, color=0x000000, head_color=0xF0F0F0, visible=False)
k3points = k3d.points(positions=[], shader="mesh", point_size=0.03125, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-1.0, +1.0])
k3proj = k3d.line(vertices=[], attribute=[], shader="thick", line_width=0.125, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-1.0, +1.0])


pladd(plot, k3proj, k3points, k3curve, k3flow)

In [ ]:
def toggle(k, vis: bool):
    k.visible = vis


toggle2.observe(lambda ch: toggle(k3flow, ch.new), "value")

In [ ]:
def rerandomize():
    global parab
    parab = randomize()
    regenerate_curve()
    regenerate_image()
    k3proj.vertices = []
    k3points.positions = []


def regenerate_curve():
    k3curve.vertices = f32(unflat_np(parab.curve(tspace)))
    k3curve.attribute = f32(tspace)
    k3curve.color_range = (-1.0, 1.0)
    k3flow.origins = garr(unflat(parab.point(t)) for t in (-1.0, 0.0, +1.0))
    k3flow.vectors = garr(normalize(unflat(parab.flow(t))) for t in (-1.0, 0.0, +1.0)) * 1.0


regenerate_curve()

In [ ]:
randomize_btn.on_click(lambda _: rerandomize())

In [ ]:
def sample_point(sample: npvec):
    sample = sample[:2]
    proj = project(parab, sample)
    k3proj.vertices = [unflat(sample), unflat(proj.pnt)]
    k3proj.attribute = [proj.sdist, proj.sdist]
    k3points.positions = [unflat(sample), unflat(proj.pnt)]
    k3points.attribute = [proj.sdist, proj.sdist]


def sample_random():
    sample_point((nprand(2) - 0.5) * 2)


sample_btn.on_click(lambda btn: sample_random())

## Texture


In [ ]:
RES = 64
PIX = 1.0 / RES
X, Y = np.meshgrid(np.arange(-0.5, 0.5, PIX), np.arange(-0.5, 0.5, PIX))
COORDS = np.stack((Y, X)).T.reshape((RES * RES, 2)) + 0.5 * PIX  # pixel centers

In [ ]:
imagedata = np.random.random((RES, RES))
k3image = k3d.texture(attribute=imagedata, interpolation=False, color_map=k3d.colormaps.matplotlib_color_maps.Seismic, color_range=[-1.0, 1.0])

pladd(plot, k3image)

In [ ]:
def regenerate_image():
    global imagedata
    imagedata = garr(project(parab, s).sdist for s in COORDS)
    k3image.attribute = imagedata.reshape((RES, RES))


regenerate_image()